# 1-D Gaussian Process Adaptive Sampling

This example builds and tunes a scalar-valued Gaussian process over a one-dimensional domain. We begin with a small set of noisy observations, tune the covariance length scale, and then add observations where the posterior variance is largest.

In [ ]:
import jax

# Un/comment this for double/single precision:
# jax.config.update('jax_enable_x64', True)

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import UncertainSCI.gp as gp


D = 1  
C = 1
SEED = 0x01234567
GP_SEED = 0xDEADBEEF
rng = np.random.default_rng(SEED)

DOMAIN = (0, 2 * jnp.pi)
NOISE_VARIANCE = 1e-2

FIGSIZE = (7, 7 / 1.6)
FIGDPI = None

def print_loss_here(g):  # Simple helper for this notebook; does not generalize.
    print(
        f'Loss at D = {jnp.squeeze(g.k.D()):.4e} (length scale {1 / jnp.sqrt(jnp.squeeze(g.k.D())):.4e}): '
        f'{g.loss_hyperparameters(train_x, train_y, train_s):.4e}'
    )

def plot_in_subplots(nrows=1, ncols=1, **kwargs):
    """
    Returns ``(fig, axes)`` with plot scaled correctly for ``(nrows, ncols)``.
    """
    if 'figsize' in kwargs:
        raise ValueError("kwargs had 'figsize' key: that's the whole point of this function!")
    return plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * FIGSIZE[0], nrows * FIGSIZE[1]),
        **kwargs
    )


## True function and noisy observations

The hidden function combines two oscillatory factors and is evaluated on the interval from zero to $2\pi$. Observations have independent Gaussian noise with a known, constant variance. `GaussianProcess.condition` expects this variance—not its square root.

In [ ]:
x_plot = jnp.linspace(*DOMAIN, 1000).reshape((-1, D))

def f_hidden(x):
    return 2 * jnp.cos(x) * jnp.sin(4 * x)

def f_noisy(x):
    y = f_hidden(x)
    noise = jnp.sqrt(NOISE_VARIANCE) * jnp.asarray(
        rng.normal(size=y.shape)
    )
    return y + noise, NOISE_VARIANCE * jnp.ones_like(y)


In [ ]:
truth = f_hidden(x_plot)
noisy, noise = f_noisy(x_plot)

fig, (ax1, ax2) = plot_in_subplots(2, 1, sharex=True, sharey=True)

ax1.plot(truth)
ax1.set_title('Truth')

ax2.plot(noisy)
ax2.set_title('Noisy View')

plt.show()


## Define the Gaussian process

The prior has a fixed zero affine mean and a Gaussian covariance kernel. The kernel parameter $D$ determines how quickly correlation decays with distance: its associated length scale is $1 / \sqrt{D}$. We leave $D$ tunable so that it can be learned from the observations.

In [ ]:
mu = gp.mean.Affine(
    dim=D,
    cdim=C,
    a=0. * jnp.ones((C, D)),
    b=0.,
    a_is_static=True,
    b_is_static=True
)
k = gp.kernel.Gaussian(
    dim=D,
    cdim=C,
    D=jnp.ones((1, 1)),
)
g = gp.GaussianProcess(dim=D, cdim=C, mu=mu, k=k, seed=GP_SEED, nugget=1e-3)


## Inspect the prior

Before conditioning, the process reflects only the chosen mean and covariance. The upper panel shows prior realizations and their mean, with the hidden function included for reference; the lower panel shows the marginal prior variance. At this point the GP has not observed the hidden function.

In [ ]:
fig, (ax1, ax2) = plot_in_subplots(2, 1, dpi=FIGDPI)
gp.vis.plot_distribution_mean(ax1, g, x_plot, f_hidden, 'prior')
gp.vis.plot_distribution_variance(ax2, g, x_plot, 'prior')
plt.show()


## Condition on initial observations

We begin with ten evenly spaced observations. Each call to `f_noisy` returns both an observed value and its noise variance, which are supplied to `condition` along with the observation coordinates.

In [ ]:
N_INIT = 10

train_x = jnp.linspace(0, 2 * jnp.pi, N_INIT).reshape(-1, D)
train_y, train_s = f_noisy(train_x)    

g.condition(train_x, train_y, train_s)


## Initial posterior

Conditioning updates the distribution while retaining the initial kernel parameter. The first two panels show the posterior distribution and marginal variance. The final panel evaluates the negative log marginal likelihood over possible values of $D$, showing where hyperparameter tuning should move the current kernel.

In [ ]:
fig, (ax1, ax2, ax3) = plot_in_subplots(3, 1, dpi=FIGDPI)
gp.vis.plot_distribution_mean(ax1, g, x_plot, f_hidden, 'posterior')
gp.vis.plot_distribution_variance(ax2, g, x_plot, 'posterior', colorlast=False)
gp.vis.plot_loss_landscape(ax3, g, ('k', 'D'), jnp.logspace(-2, 2, 1000))
plt.show()

print_loss_here(g)


## Tune the kernel hyperparameter

`tune` minimizes the negative log marginal likelihood with respect to the non-static GP parameters. Here that means learning $D$, and therefore the covariance length scale, from the initial observations. The optimization trace below shows how the loss changes with each step.

In [ ]:
losses = g.tune()

plt.figure(figsize=FIGSIZE)
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.show()


## Posterior after tuning

Replotting the same diagnostics makes the effect of tuning visible. The learned length scale changes both the posterior interpolation and how rapidly uncertainty grows between observations, while the loss landscape locates the tuned parameter relative to nearby alternatives.

In [ ]:
fig, (ax1, ax2, ax3) = plot_in_subplots(3, 1, dpi=FIGDPI)
gp.vis.plot_distribution_mean(ax1, g, x_plot, f_hidden, 'posterior')
gp.vis.plot_distribution_variance(ax2, g, x_plot, 'posterior', colorlast=False)
gp.vis.plot_loss_landscape(ax3, g, ('k', 'D'), jnp.logspace(-2, 2, 1000))
plt.show()

print_loss_here(g)


## Adaptively add observations

At each iteration, we first select the best point from a coarse candidate grid using marginal posterior variance. Starting from that candidate, `get_sample_point` directly maximizes posterior variance within the domain. We observe the hidden function there, condition on all available data, and tune the kernel again. In each variance plot, the newly added coordinate is highlighted separately from the earlier observations.

In [ ]:
N_TRAIN = 10
N_SAMPLE = 20

x_sample = jnp.linspace(0, 2 * jnp.pi, N_SAMPLE).reshape(-1, D)

for i in range(N_TRAIN):
    # Greedily sample posterior variance:
    _v = jnp.diag(g.posterior_covariance(x_sample, x_sample))
    _x = x_sample[jnp.argmax(_v)].reshape(-1, D)

    # Iteratively optimize posterior variance:
    _x = g.get_sample_point(_x.reshape((1,)), ranges=np.atleast_2d(DOMAIN).T.tolist())

    # Observe that point:
    _y, _s = f_noisy(_x)

    train_x = jnp.concat((train_x, _x))
    train_y = jnp.concat((train_y, _y))
    train_s = jnp.concat((train_s, _s))

    # Condition with new data:
    g.condition(train_x, train_y, train_s)
    g.tune()

    # Plot:
    fig, (ax1, ax2, ax3) = plot_in_subplots(3, 1, dpi=FIGDPI)
    gp.vis.plot_distribution_mean(ax1, g, x_plot, f_hidden, 'posterior')
    gp.vis.plot_distribution_variance(ax2, g, x_plot, 'posterior')
    gp.vis.plot_loss_landscape(ax3, g, ('k', 'D'), jnp.logspace(-2, 2, 1000))
    plt.show()
    print_loss_here(g)
    print('\n' * 3)


## Result

Adaptive sampling directs new observations toward regions where the current model is least certain. Reconditioning reduces uncertainty near each new observation, while repeated tuning updates the covariance length scale as more information becomes available.